In [1]:
# Step 1: Ensure all libraries are installed
!pip install rouge-score nltk bert-score -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00


In [3]:
import pandas as pd
import os
import torch
from rouge_score import rouge_scorer
from bert_score import score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction # Neu hinzugefügt

# Step 2: Configuration
data_dir = "."
file_clean = os.path.join(data_dir, "dataset_clean.csv")
file_gold = os.path.join(data_dir, "gold_standard.csv")
file_m1 = os.path.join(data_dir, "model_1_inference_groq.csv")
file_m2 = os.path.join(data_dir, "model_2_results_final.csv")
file_m3 = os.path.join(data_dir, "model_3_rag_tinyllama.csv")

def run_benchmarking_evaluation():
    try:
        # Load datasets
        clean_df = pd.read_csv(file_clean)
        gold_df = pd.read_csv(file_gold)
        m1_df = pd.read_csv(file_m1).set_index('id')
        m2_df = pd.read_csv(file_m2).set_index('id')
        m3_df = pd.read_csv(file_m3).set_index('id')

        # ID Repair logic
        prompt_to_id = {str(row['prompt']).strip().lower(): row['id'] for _, row in clean_df.iterrows()}
        gold_df['id'] = gold_df.apply(lambda r: prompt_to_id.get(str(r['prompt']).strip().lower(), r['id']), axis=1)
        gold_ref = gold_df.set_index('id')

        common_ids = m1_df.index.intersection(m2_df.index).intersection(m3_df.index).intersection(gold_ref.index)
        device = "cuda" if torch.cuda.is_available() else "cpu"

        # Initialize scorers
        r_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        smoothie = SmoothingFunction().method1 # Für BLEU

        all_refs = []
        candidates = {"M1": [], "M2": [], "M3": []}
        results_list = []

        for idx in common_ids:
            ref = str(gold_ref.loc[idx, 'correct_answer'])
            if isinstance(ref, pd.Series): ref = ref.iloc[0]
            all_refs.append(ref)

            res_row = {'id': idx}
            for m_key, df in [("M1", m1_df), ("M2", m2_df), ("M3", m3_df)]:
                cand = str(df.loc[idx, 'answer'])
                candidates[m_key].append(cand)

                # NEW: Calculate BLEU
                res_row[f'{m_key}_BLEU'] = sentence_bleu([ref.split()], cand.split(), smoothing_function=smoothie)

                # Calculate ROUGE metrics
                scores = r_scorer.score(ref, cand)
                res_row[f'{m_key}_R1'] = scores['rouge1'].fmeasure
                res_row[f'{m_key}_R2'] = scores['rouge2'].fmeasure
                res_row[f'{m_key}_RL'] = scores['rougeL'].fmeasure
            results_list.append(res_row)

        # BERTScore calculation
        bert_f1_scores = {}
        for m_key in ["M1", "M2", "M3"]:
            P, R, F1 = score(candidates[m_key], all_refs, lang="de", device=device, verbose=False)
            bert_f1_scores[m_key] = F1.mean().item()

        # Final Summary Table
        df_res = pd.DataFrame(results_list)
        summary = pd.DataFrame({
            'Model': ['Model 1 (Groq 70B)', 'Model 2 (Llama-3 8B FT)', 'Model 3 (TinyLlama RAG)'],
            'BLEU': [df_res['M1_BLEU'].mean(), df_res['M2_BLEU'].mean(), df_res['M3_BLEU'].mean()],
            'ROUGE-1 F1': [df_res['M1_R1'].mean(), df_res['M2_R1'].mean(), df_res['M3_R1'].mean()],
            'ROUGE-2 F1': [df_res['M1_R2'].mean(), df_res['M2_R2'].mean(), df_res['M3_R2'].mean()],
            'ROUGE-L F1': [df_res['M1_RL'].mean(), df_res['M2_RL'].mean(), df_res['M3_RL'].mean()],
            'BERTScore F1': [bert_f1_scores['M1'], bert_f1_scores['M2'], bert_f1_scores['M3']]
        })

        print("\n=== FINAL BENCHMARK RESULTS ===")
        print(summary.to_string(index=False))

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    run_benchmarking_evaluation()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== FINAL BENCHMARK RESULTS ===
                  Model     BLEU  ROUGE-1 F1  ROUGE-2 F1  ROUGE-L F1  BERTScore F1
     Model 1 (Groq 70B) 0.018738    0.214681    0.058439    0.137936      0.699683
Model 2 (Llama-3 8B FT) 0.000977    0.018558    0.003192    0.017660      0.522058
Model 3 (TinyLlama RAG) 0.005732    0.058150    0.015940    0.044965      0.634115
